In [1]:
!pip install mediapipe lightgbm scikit-learn pillow tqdm matplotlib pandas numpy opencv-headless mediapipe

^C


In [ ]:
## FF++ balanced -- oversampling; creating a augmented set of real images
!pip install lightgbm scikit-learn pillow tqdm matplotlib pandas numpy

import os, time, random, warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================================
# CONFIG
# ============================================================

METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

@dataclass
class Config:
    image_size: int = 224
    resolutions: tuple = (128, 224, 256, 384)

    small_block_size: int = 13
    large_block_size: int = 32
    patch_size: int = 3
    stride: int = 2

    max_train_fit_per_class: int = 1500
    dft_num_splits: int = 31

    landmark_keep_ratio: float = 0.35
    region_keep_ratio: float = 0.15

    spatial_pca_energy: float = 0.80
    spatial_pca_max_channels: int = 10

    uncertain_low: float = 0.4
    uncertain_high: float = 0.6

    lgb_num_leaves: int = 64
    lgb_n_estimators: int = 300
    lgb_learning_rate: float = 0.05
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8

    random_state: int = 42
    out_dir: str = "./outputs_defakehoppp_ffpp_oversampled"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)

random.seed(cfg.random_state)
np.random.seed(cfg.random_state)

# ============================================================
# DATASET
# ============================================================

def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)

    required = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
    df = df.rename(columns={"frame_path": "image_path"})

    print("=" * 70)
    print("FF++ DATASET LOADED")
    print("=" * 70)
    print("Total usable frames:", len(df))

    for split in ["Train", "Val", "Test"]:
        sdf = df[df["split"].str.lower() == split.lower()]
        real = int((sdf["label"] == 0).sum())
        fake = int((sdf["label"] == 1).sum())
        videos = sdf["video_id"].nunique()
        print(f"{split}: real={real}, fake={fake}, total={len(sdf)}, videos={videos}")

    return df

def make_oversampled_train_df(train_df, seed=42):
    real_df = train_df[train_df["label"] == 0].copy()
    fake_df = train_df[train_df["label"] == 1].copy()

    real_os = real_df.sample(
        n=len(fake_df),
        replace=True,
        random_state=seed
    ).copy()

    real_os["is_oversampled_real"] = True
    fake_df["is_oversampled_real"] = False

    oversampled_df = (
        pd.concat([real_os, fake_df])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    print("\nOversampled train set")
    print(oversampled_df["label_name"].value_counts())
    print("Total oversampled train samples:", len(oversampled_df))
    print("Oversampled real rows:", int(oversampled_df["is_oversampled_real"].sum()))

    return oversampled_df

df_meta = load_ffpp_metadata(METADATA_CSV)

train_df_full = df_meta[df_meta["split"].str.lower() == "train"].reset_index(drop=True)
val_df = df_meta[df_meta["split"].str.lower() == "val"].reset_index(drop=True)
test_df = df_meta[df_meta["split"].str.lower() == "test"].reset_index(drop=True)

train_df = make_oversampled_train_df(train_df_full, seed=cfg.random_state)

print("\nFinal split sizes used")
print("Train original:", len(train_df_full))
print("Train oversampled:", len(train_df))
print("Val original:", len(val_df))
print("Test original:", len(test_df))

# ============================================================
# IMAGE HELPERS
# ============================================================

def load_image_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def resize_image_np(img, size):
    return np.array(Image.fromarray(img).resize((size, size)))

def crop_square_block(img_rgb, center_xy, block_size):
    h, w = img_rgb.shape[:2]
    cx, cy = center_xy
    half = block_size // 2

    x1, y1 = cx - half, cy - half
    x2, y2 = x1 + block_size, y1 + block_size

    out = np.zeros((block_size, block_size, 3), dtype=np.uint8)

    sx1, sy1 = max(0, x1), max(0, y1)
    sx2, sy2 = min(w, x2), min(h, y2)

    dx1, dy1 = sx1 - x1, sy1 - y1
    dx2, dy2 = dx1 + (sx2 - sx1), dy1 + (sy2 - sy1)

    out[dy1:dy2, dx1:dx2] = img_rgb[sy1:sy2, sx1:sx2]
    return out

# ============================================================
# DEFAKEHOP++ APPROX BLOCKS
# ============================================================

SMALL_BLOCK_CENTERS = [
    (0.32, 0.28), (0.45, 0.30), (0.68, 0.28), (0.55, 0.30),
    (0.38, 0.45), (0.62, 0.45), (0.50, 0.40), (0.50, 0.58),
]

LARGE_BLOCK_CENTERS = [
    (0.35, 0.35), (0.65, 0.35), (0.50, 0.68),
]

def extract_defakehoppp_blocks(img_rgb):
    h, w = img_rgb.shape[:2]

    small_blocks = []
    for nx, ny in SMALL_BLOCK_CENTERS:
        small_blocks.append(
            crop_square_block(
                img_rgb,
                (int(nx * w), int(ny * h)),
                cfg.small_block_size,
            )
        )

    large_blocks = []
    for nx, ny in LARGE_BLOCK_CENTERS:
        large_blocks.append(
            crop_square_block(
                img_rgb,
                (int(nx * w), int(ny * h)),
                cfg.large_block_size,
            )
        )

    return small_blocks, large_blocks

def extract_patches_from_block(block, patch_size=3, stride=1):
    h, w, c = block.shape
    patches = []

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patches.append(block[y:y+patch_size, x:x+patch_size, :].reshape(-1))

    return np.array(patches, dtype=np.float32)

# ============================================================
# PIXELHOP PCA
# ============================================================

def collect_training_patches(train_df, image_size=224):
    real_paths = train_df[train_df["label"] == 0]["image_path"].tolist()[:cfg.max_train_fit_per_class]
    fake_paths = train_df[train_df["label"] == 1]["image_path"].tolist()[:cfg.max_train_fit_per_class]
    selected_paths = real_paths + fake_paths

    patches = []

    for path in tqdm(selected_paths, desc="Collecting training patches"):
        try:
            img = load_image_rgb(path)
            img = resize_image_np(img, image_size)
            small_blocks, large_blocks = extract_defakehoppp_blocks(img)

            for block in small_blocks + large_blocks:
                p = extract_patches_from_block(block, cfg.patch_size, stride=1)
                if len(p) > 0:
                    patches.append(p)

        except Exception:
            continue

    return np.concatenate(patches, axis=0)

def fit_pixelhop_pca(train_df, image_size=224):
    patches = collect_training_patches(train_df, image_size=image_size)
    pca = PCA(n_components=27, random_state=cfg.random_state)
    pca.fit(patches)
    return pca

def pixelhop_transform_block(block, pca_model):
    h, w, c = block.shape
    out_h = (h - cfg.patch_size) // cfg.stride + 1
    out_w = (w - cfg.patch_size) // cfg.stride + 1

    feats = np.zeros((out_h, out_w, 27), dtype=np.float32)

    oy = 0
    for y in range(0, h - cfg.patch_size + 1, cfg.stride):
        ox = 0
        for x in range(0, w - cfg.patch_size + 1, cfg.stride):
            patch = block[y:y+cfg.patch_size, x:x+cfg.patch_size, :].reshape(1, -1).astype(np.float32)
            feats[oy, ox, :] = pca_model.transform(patch)[0]
            ox += 1
        oy += 1

    return feats

def extract_all_fmaps_for_image(img_rgb, pixelhop_pca):
    small_blocks, large_blocks = extract_defakehoppp_blocks(img_rgb)
    small_fmaps = [pixelhop_transform_block(b, pixelhop_pca) for b in small_blocks]
    large_fmaps = [pixelhop_transform_block(b, pixelhop_pca) for b in large_blocks]
    return small_fmaps, large_fmaps

# ============================================================
# SPATIAL PCA
# ============================================================

def fit_spatial_pca_for_blocktype(feature_maps_list):
    c = feature_maps_list[0].shape[-1]

    variances = []
    for ch in range(c):
        vals = [np.var(fmap[:, :, ch]) for fmap in feature_maps_list]
        variances.append(np.mean(vals))

    ranked = np.argsort(variances)[::-1]
    kept = ranked[:cfg.spatial_pca_max_channels]

    channel_models = {}
    kept_channels = []

    for ch in kept:
        X = np.stack([fmap[:, :, ch].reshape(-1) for fmap in feature_maps_list], axis=0)

        pca = PCA(random_state=cfg.random_state)
        pca.fit(X)

        cum = np.cumsum(pca.explained_variance_ratio_)
        n_keep = int(np.searchsorted(cum, cfg.spatial_pca_energy) + 1)
        n_keep = max(1, n_keep)

        channel_models[ch] = {
            "mean": pca.mean_.copy(),
            "components": pca.components_[:n_keep].copy(),
        }
        kept_channels.append(ch)

    return {
        "kept_channels": kept_channels,
        "channel_models": channel_models,
    }

def apply_spatial_pca_to_block(fmap, spatial_pca_model):
    feats = []

    for ch in spatial_pca_model["kept_channels"]:
        vec = fmap[:, :, ch].reshape(1, -1)
        mean = spatial_pca_model["channel_models"][ch]["mean"]
        comps = spatial_pca_model["channel_models"][ch]["components"]
        proj = (vec - mean) @ comps.T
        feats.append(proj[0])

    return np.concatenate(feats, axis=0).astype(np.float32)

def collect_feature_maps_for_training(df, image_size, pixelhop_pca):
    all_small = [[] for _ in range(8)]
    all_large = [[] for _ in range(3)]
    labels = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Collecting feature maps"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            small_fmaps, large_fmaps = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                all_small[i].append(small_fmaps[i])
            for i in range(3):
                all_large[i].append(large_fmaps[i])

            labels.append(int(row["label"]))
        except Exception:
            continue

    return all_small, all_large, np.array(labels, dtype=np.int32)

def fit_all_spatial_pca_models(train_df, image_size, pixelhop_pca):
    all_small, all_large, labels = collect_feature_maps_for_training(
        train_df,
        image_size,
        pixelhop_pca,
    )

    small_models = []
    for i in range(8):
        small_models.append(fit_spatial_pca_for_blocktype(all_small[i]))

    large_models = []
    for i in range(3):
        large_models.append(fit_spatial_pca_for_blocktype(all_large[i]))

    return small_models, large_models

# ============================================================
# FEATURE MATRIX
# ============================================================

def build_feature_matrix(df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models):
    X_small = [[] for _ in range(8)]
    X_large = [[] for _ in range(3)]
    y = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Build features @ {image_size}"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            small_fmaps, large_fmaps = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                X_small[i].append(apply_spatial_pca_to_block(small_fmaps[i], small_spatial_models[i]))

            for i in range(3):
                X_large[i].append(apply_spatial_pca_to_block(large_fmaps[i], large_spatial_models[i]))

            y.append(int(row["label"]))
        except Exception:
            continue

    X_small = [np.stack(v, axis=0) for v in X_small]
    X_large = [np.stack(v, axis=0) for v in X_large]
    y = np.array(y, dtype=np.int32)

    return X_small, X_large, y

# ============================================================
# DFT FEATURE SELECTION
# ============================================================

def binary_cross_entropy_from_probs(y_true, p):
    eps = 1e-8
    p = np.clip(p, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)).mean()

def dft_score_feature(x, y):
    x = np.asarray(x).astype(np.float32)
    y = np.asarray(y).astype(np.int32)

    xmin, xmax = x.min(), x.max()
    if xmax <= xmin:
        return 1e9, None

    splits = np.linspace(xmin, xmax, cfg.dft_num_splits + 2)[1:-1]

    best_ce = 1e9
    best_split = None

    for s in splits:
        left = x <= s
        right = ~left

        if left.sum() == 0 or right.sum() == 0:
            continue

        p_left = y[left].mean()
        p_right = y[right].mean()
        probs = np.where(left, p_left, p_right)

        ce = binary_cross_entropy_from_probs(y, probs)

        if ce < best_ce:
            best_ce = ce
            best_split = s

    return best_ce, best_split

def fit_dft_selector(X, y, keep_ratio):
    scores = []
    splits = []

    for j in tqdm(range(X.shape[1]), desc="DFT"):
        ce, s = dft_score_feature(X[:, j], y)
        scores.append(ce)
        splits.append(s)

    scores = np.array(scores)
    order = np.argsort(scores)
    k = max(1, int(len(order) * keep_ratio))
    selected = order[:k]

    return {
        "selected_indices": selected,
        "scores": scores,
        "splits": splits,
    }

def apply_dft_selector(X, selector):
    return X[:, selector["selected_indices"]]

# ============================================================
# TRAIN PIPELINE
# ============================================================

def fit_defakehoppp_pipeline(train_df, val_df, image_size=224):
    t0 = time.time()

    print("\nFitting PixelHop PCA...")
    pixelhop_pca = fit_pixelhop_pca(train_df, image_size=image_size)
    t1 = time.time()

    print("\nFitting Spatial PCA...")
    small_spatial_models, large_spatial_models = fit_all_spatial_pca_models(
        train_df,
        image_size=image_size,
        pixelhop_pca=pixelhop_pca,
    )
    t2 = time.time()

    print("\nBuilding train and validation features...")
    Xs_train, Xl_train, y_train = build_feature_matrix(
        train_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )

    Xs_val, Xl_val, y_val = build_feature_matrix(
        val_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )
    t3 = time.time()

    print("\nRunning DFT feature selection...")
    small_selectors = []
    Xs_train_sel = []
    Xs_val_sel = []

    for i in range(8):
        sel = fit_dft_selector(Xs_train[i], y_train, cfg.landmark_keep_ratio)
        small_selectors.append(sel)
        Xs_train_sel.append(apply_dft_selector(Xs_train[i], sel))
        Xs_val_sel.append(apply_dft_selector(Xs_val[i], sel))

    large_selectors = []
    Xl_train_sel = []
    Xl_val_sel = []

    for i in range(3):
        sel = fit_dft_selector(Xl_train[i], y_train, cfg.region_keep_ratio)
        large_selectors.append(sel)
        Xl_train_sel.append(apply_dft_selector(Xl_train[i], sel))
        Xl_val_sel.append(apply_dft_selector(Xl_val[i], sel))

    X_train = np.concatenate(Xs_train_sel + Xl_train_sel, axis=1)
    X_val = np.concatenate(Xs_val_sel + Xl_val_sel, axis=1)
    t4 = time.time()

    print("\nTraining LightGBM classifier...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = lgb.LGBMClassifier(
        objective="binary",
        num_leaves=cfg.lgb_num_leaves,
        n_estimators=cfg.lgb_n_estimators,
        learning_rate=cfg.lgb_learning_rate,
        subsample=cfg.lgb_subsample,
        colsample_bytree=cfg.lgb_colsample_bytree,
        random_state=cfg.random_state,
    )

    clf.fit(X_train, y_train)
    t5 = time.time()

    timing = {
        "pixelhop_fit_sec": t1 - t0,
        "spatial_pca_fit_sec": t2 - t1,
        "feature_build_sec": t3 - t2,
        "dft_sec": t4 - t3,
        "classifier_sec": t5 - t4,
        "total_train_sec": t5 - t0,
    }

    model = {
        "pixelhop_pca": pixelhop_pca,
        "small_spatial_models": small_spatial_models,
        "large_spatial_models": large_spatial_models,
        "small_selectors": small_selectors,
        "large_selectors": large_selectors,
        "scaler": scaler,
        "clf": clf,
        "image_size": image_size,
    }

    return model, timing

# ============================================================
# EVALUATION
# ============================================================

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    uncertain_mask = (y_prob > cfg.uncertain_low) & (y_prob < cfg.uncertain_high)
    uncertain_rate = uncertain_mask.mean() * 100.0
    avg_uncertainty = (1.0 - 2.0 * np.abs(y_prob - 0.5)).mean()

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "avg_uncertainty": avg_uncertainty,
    }

def transform_dataframe_with_model(df, model, image_size=None):
    if image_size is None:
        image_size = model["image_size"]

    Xs, Xl, y = build_feature_matrix(
        df,
        image_size,
        model["pixelhop_pca"],
        model["small_spatial_models"],
        model["large_spatial_models"],
    )

    Xs_sel = [
        apply_dft_selector(Xs[i], model["small_selectors"][i])
        for i in range(8)
    ]

    Xl_sel = [
        apply_dft_selector(Xl[i], model["large_selectors"][i])
        for i in range(3)
    ]

    X = np.concatenate(Xs_sel + Xl_sel, axis=1)
    X = model["scaler"].transform(X)

    return X, y

def evaluate_model(model, df, image_size=None, model_name="DefakeHop++ Approx FF++ Oversampled"):
    X, y = transform_dataframe_with_model(df, model, image_size=image_size)
    probs = model["clf"].predict_proba(X)[:, 1]

    m = compute_metrics(y, probs)

    return pd.DataFrame([{
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "AUC": m["auc"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Uncertain Rate (%)": m["uncertain_rate"],
        "Avg Uncertainty": m["avg_uncertainty"],
    }])

def evaluate_resolutions(model, test_df):
    rows = []

    for res in cfg.resolutions:
        X, y = transform_dataframe_with_model(test_df, model, image_size=res)
        probs = model["clf"].predict_proba(X)[:, 1]
        m = compute_metrics(y, probs)

        rows.append({
            "Resolution": res,
            "Accuracy": m["accuracy"],
            "AUC": m["auc"],
            "Fast (%)": 0.00,
            "Medium (%)": 0.00,
            "Full (%)": 100.00,
            "Avg Uncertainty": m["avg_uncertainty"],
            "Uncertain Rate (%)": m["uncertain_rate"],
        })

    return pd.DataFrame(rows)

def measure_inference_time(model, df, image_size, max_samples=500):
    sub_df = df.iloc[:max_samples].copy()

    start = time.time()
    X, y = transform_dataframe_with_model(sub_df, model, image_size=image_size)
    _ = model["clf"].predict_proba(X)[:, 1]
    elapsed = time.time() - start

    n = len(sub_df)
    return {
        "Mode": f"{image_size}x{image_size}",
        "Time per Image (ms)": (elapsed / max(n, 1)) * 1000.0,
        "FPS": n / max(elapsed, 1e-8),
    }

def measure_inference_time_by_resolution(model, test_df):
    rows = []
    for res in cfg.resolutions:
        rows.append(measure_inference_time(model, test_df, res, max_samples=500))
    return pd.DataFrame(rows)

def plot_resolution_results(res_df):
    tmp = res_df.sort_values("Resolution")

    plt.figure(figsize=(8, 4))
    plt.plot(tmp["Resolution"], tmp["Accuracy"], marker="o", label="Accuracy")
    plt.plot(tmp["Resolution"], tmp["AUC"], marker="s", label="AUC")
    plt.plot(tmp["Resolution"], tmp["Avg Uncertainty"], marker="^", label="Avg Uncertainty")
    plt.xlabel("Resolution")
    plt.ylabel("Score")
    plt.title("DefakeHop++ Approx FF++ Oversampled Train vs Resolution")
    plt.grid(True)
    plt.legend()
    plt.show()

# ============================================================
# RUN
# ============================================================

print("\n" + "=" * 70)
print("RUNNING DEFAKEHOP++ APPROX ON FF++ WITH REAL OVERSAMPLING")
print("=" * 70)

model, timing = fit_defakehoppp_pipeline(
    train_df=train_df,
    val_df=val_df,
    image_size=cfg.image_size,
)

test_results_df = evaluate_model(
    model,
    test_df,
    image_size=cfg.image_size,
    model_name="DefakeHop++ Approx FF++ Real Oversampled Train",
)

res_df = evaluate_resolutions(model, test_df)
inf_df = measure_inference_time_by_resolution(model, test_df)

train_time_df = pd.DataFrame([{
    "Model Type": "DefakeHop++ Approx FF++ Real Oversampled Train",
    "mins": timing["total_train_sec"] / 60.0,
    "hrs": timing["total_train_sec"] / 3600.0,
}])

dataset_summary = pd.DataFrame([{
    "Experiment Name": "DefakeHop++ Approx Real Oversampled Train",
    "Dataset": "FaceForensics++ C23",
    "Train Samples Original": len(train_df_full),
    "Train Samples Used Oversampled": len(train_df),
    "Validation Samples": len(val_df),
    "Test Samples": len(test_df),
    "Train Real Used": int((train_df["label"] == 0).sum()),
    "Train Fake Used": int((train_df["label"] == 1).sum()),
    "Validation Real": int((val_df["label"] == 0).sum()),
    "Validation Fake": int((val_df["label"] == 1).sum()),
    "Test Real": int((test_df["label"] == 0).sum()),
    "Test Fake": int((test_df["label"] == 1).sum()),
    "Balancing Method": "Train-only oversampling of real class with replacement to match fake class",
    "Resolutions Used": str(cfg.resolutions),
    "Input Modality": "RGB handcrafted features",
    "Image Size": cfg.image_size,
}])

print("\nTest Results")
display(test_results_df)

print("\nAccuracy Based on Resolution")
display(res_df)

print("\nInference Time")
display(inf_df)

print("\nTraining Time")
display(train_time_df)

print("\nExperiment Summary")
display(dataset_summary)

plot_resolution_results(res_df)

# ============================================================
# SAVE
# ============================================================

test_results_df.to_csv(os.path.join(cfg.out_dir, "test_results.csv"), index=False)
res_df.to_csv(os.path.join(cfg.out_dir, "accuracy_based_on_resolution.csv"), index=False)
inf_df.to_csv(os.path.join(cfg.out_dir, "inference_time.csv"), index=False)
train_time_df.to_csv(os.path.join(cfg.out_dir, "training_time.csv"), index=False)
dataset_summary.to_csv(os.path.join(cfg.out_dir, "dataset_summary.csv"), index=False)
pd.DataFrame([timing]).to_csv(os.path.join(cfg.out_dir, "training_stage_timing_breakdown.csv"), index=False)

print("\nSaved outputs to:", cfg.out_dir)

  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached pillow-12.2.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached matplotlib-3.10.9-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-no

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FF++ DATASET LOADED
Total usable frames: 55953
Train: real=6400, fake=38356, total=44756, videos=5600
Val: real=800, fake=4800, total=5600, videos=700
Test: real=800, fake=4797, total=5597, videos=700

Oversampled train set
label_name
Fake    38356
Real    38356
Name: count, dtype: int64
Total oversampled train samples: 76712
Oversampled real rows: 38356

Final split sizes used
Train original: 44756
Train oversampled: 76712
Val original: 5600
Test original: 5597

RUNNING DEFAKEHOP++ APPROX ON FF++ WITH REAL OVERSAMPLING

Fitting PixelHop PCA...



Fitting Spatial PCA...


In [ ]:
#FF++ Balanced -- Undersampling
!pip install lightgbm scikit-learn pillow tqdm matplotlib pandas numpy

import os, io, time, random, warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================================
# CONFIG
# ============================================================

METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

@dataclass
class Config:
    image_size: int = 224
    resolutions: tuple = (128, 224, 256, 384)

    small_block_size: int = 13
    large_block_size: int = 32
    patch_size: int = 3
    stride: int = 2

    max_train_fit_per_class: int = 1500
    dft_num_splits: int = 31

    landmark_keep_ratio: float = 0.35
    region_keep_ratio: float = 0.15

    spatial_pca_energy: float = 0.80
    spatial_pca_max_channels: int = 10

    uncertain_low: float = 0.4
    uncertain_high: float = 0.6

    lgb_num_leaves: int = 64
    lgb_n_estimators: int = 300
    lgb_learning_rate: float = 0.05
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8

    random_state: int = 42
    out_dir: str = "./outputs_defakehoppp_ffpp_balanced"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)

random.seed(cfg.random_state)
np.random.seed(cfg.random_state)

# ============================================================
# DATASET
# ============================================================

def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)

    required = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
    df = df.rename(columns={"frame_path": "image_path"})

    print("=" * 70)
    print("FF++ DATASET LOADED")
    print("=" * 70)
    print("Total usable frames:", len(df))

    for split in ["Train", "Val", "Test"]:
        sdf = df[df["split"].str.lower() == split.lower()]
        real = int((sdf["label"] == 0).sum())
        fake = int((sdf["label"] == 1).sum())
        videos = sdf["video_id"].nunique()
        print(f"{split}: real={real}, fake={fake}, total={len(sdf)}, videos={videos}")

    return df

def make_balanced_train_df(train_df, seed=42):
    real_df = train_df[train_df["label"] == 0]
    fake_df = train_df[train_df["label"] == 1]

    n = min(len(real_df), len(fake_df))

    real_bal = real_df.sample(n=n, random_state=seed)
    fake_bal = fake_df.sample(n=n, random_state=seed)

    balanced_df = (
        pd.concat([real_bal, fake_bal])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    print("\nBalanced train set by undersampling")
    print(balanced_df["label_name"].value_counts())
    print("Total balanced train samples:", len(balanced_df))

    return balanced_df

df_meta = load_ffpp_metadata(METADATA_CSV)

train_df_full = df_meta[df_meta["split"].str.lower() == "train"].reset_index(drop=True)
val_df = df_meta[df_meta["split"].str.lower() == "val"].reset_index(drop=True)
test_df = df_meta[df_meta["split"].str.lower() == "test"].reset_index(drop=True)

train_df = make_balanced_train_df(train_df_full, seed=cfg.random_state)

print("\nFinal split sizes used")
print("Train balanced:", len(train_df))
print("Val original:", len(val_df))
print("Test original:", len(test_df))

# ============================================================
# IMAGE HELPERS
# ============================================================

def load_image_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def resize_image_np(img, size):
    return np.array(Image.fromarray(img).resize((size, size)))

def crop_square_block(img_rgb, center_xy, block_size):
    h, w = img_rgb.shape[:2]
    cx, cy = center_xy
    half = block_size // 2

    x1, y1 = cx - half, cy - half
    x2, y2 = x1 + block_size, y1 + block_size

    out = np.zeros((block_size, block_size, 3), dtype=np.uint8)

    sx1, sy1 = max(0, x1), max(0, y1)
    sx2, sy2 = min(w, x2), min(h, y2)

    dx1, dy1 = sx1 - x1, sy1 - y1
    dx2, dy2 = dx1 + (sx2 - sx1), dy1 + (sy2 - sy1)

    out[dy1:dy2, dx1:dx2] = img_rgb[sy1:sy2, sx1:sx2]
    return out

# ============================================================
# DEFAKEHOP++ APPROX BLOCKS
# ============================================================

SMALL_BLOCK_CENTERS = [
    (0.32, 0.28), (0.45, 0.30), (0.68, 0.28), (0.55, 0.30),
    (0.38, 0.45), (0.62, 0.45), (0.50, 0.40), (0.50, 0.58),
]

LARGE_BLOCK_CENTERS = [
    (0.35, 0.35), (0.65, 0.35), (0.50, 0.68),
]

def extract_defakehoppp_blocks(img_rgb):
    h, w = img_rgb.shape[:2]

    small_blocks = []
    for nx, ny in SMALL_BLOCK_CENTERS:
        small_blocks.append(
            crop_square_block(
                img_rgb,
                (int(nx * w), int(ny * h)),
                cfg.small_block_size,
            )
        )

    large_blocks = []
    for nx, ny in LARGE_BLOCK_CENTERS:
        large_blocks.append(
            crop_square_block(
                img_rgb,
                (int(nx * w), int(ny * h)),
                cfg.large_block_size,
            )
        )

    return small_blocks, large_blocks

def extract_patches_from_block(block, patch_size=3, stride=1):
    h, w, c = block.shape
    patches = []

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patches.append(block[y:y+patch_size, x:x+patch_size, :].reshape(-1))

    return np.array(patches, dtype=np.float32)

# ============================================================
# PIXELHOP PCA
# ============================================================

def collect_training_patches(train_df, image_size=224):
    real_paths = train_df[train_df["label"] == 0]["image_path"].tolist()[:cfg.max_train_fit_per_class]
    fake_paths = train_df[train_df["label"] == 1]["image_path"].tolist()[:cfg.max_train_fit_per_class]
    selected_paths = real_paths + fake_paths

    patches = []

    for path in tqdm(selected_paths, desc="Collecting training patches"):
        try:
            img = load_image_rgb(path)
            img = resize_image_np(img, image_size)
            small_blocks, large_blocks = extract_defakehoppp_blocks(img)

            for block in small_blocks + large_blocks:
                p = extract_patches_from_block(block, cfg.patch_size, stride=1)
                if len(p) > 0:
                    patches.append(p)

        except Exception:
            continue

    return np.concatenate(patches, axis=0)

def fit_pixelhop_pca(train_df, image_size=224):
    patches = collect_training_patches(train_df, image_size=image_size)
    pca = PCA(n_components=27, random_state=cfg.random_state)
    pca.fit(patches)
    return pca

def pixelhop_transform_block(block, pca_model):
    h, w, c = block.shape
    out_h = (h - cfg.patch_size) // cfg.stride + 1
    out_w = (w - cfg.patch_size) // cfg.stride + 1

    feats = np.zeros((out_h, out_w, 27), dtype=np.float32)

    oy = 0
    for y in range(0, h - cfg.patch_size + 1, cfg.stride):
        ox = 0
        for x in range(0, w - cfg.patch_size + 1, cfg.stride):
            patch = block[y:y+cfg.patch_size, x:x+cfg.patch_size, :].reshape(1, -1).astype(np.float32)
            feats[oy, ox, :] = pca_model.transform(patch)[0]
            ox += 1
        oy += 1

    return feats

def extract_all_fmaps_for_image(img_rgb, pixelhop_pca):
    small_blocks, large_blocks = extract_defakehoppp_blocks(img_rgb)

    small_fmaps = [pixelhop_transform_block(b, pixelhop_pca) for b in small_blocks]
    large_fmaps = [pixelhop_transform_block(b, pixelhop_pca) for b in large_blocks]

    return small_fmaps, large_fmaps

# ============================================================
# SPATIAL PCA
# ============================================================

def fit_spatial_pca_for_blocktype(feature_maps_list):
    c = feature_maps_list[0].shape[-1]

    variances = []
    for ch in range(c):
        vals = [np.var(fmap[:, :, ch]) for fmap in feature_maps_list]
        variances.append(np.mean(vals))

    ranked = np.argsort(variances)[::-1]
    kept = ranked[:cfg.spatial_pca_max_channels]

    channel_models = {}
    kept_channels = []

    for ch in kept:
        X = np.stack([fmap[:, :, ch].reshape(-1) for fmap in feature_maps_list], axis=0)

        pca = PCA(random_state=cfg.random_state)
        pca.fit(X)

        cum = np.cumsum(pca.explained_variance_ratio_)
        n_keep = int(np.searchsorted(cum, cfg.spatial_pca_energy) + 1)
        n_keep = max(1, n_keep)

        channel_models[ch] = {
            "mean": pca.mean_.copy(),
            "components": pca.components_[:n_keep].copy(),
        }
        kept_channels.append(ch)

    return {
        "kept_channels": kept_channels,
        "channel_models": channel_models,
    }

def apply_spatial_pca_to_block(fmap, spatial_pca_model):
    feats = []

    for ch in spatial_pca_model["kept_channels"]:
        vec = fmap[:, :, ch].reshape(1, -1)
        mean = spatial_pca_model["channel_models"][ch]["mean"]
        comps = spatial_pca_model["channel_models"][ch]["components"]
        proj = (vec - mean) @ comps.T
        feats.append(proj[0])

    return np.concatenate(feats, axis=0).astype(np.float32)

def collect_feature_maps_for_training(df, image_size, pixelhop_pca):
    all_small = [[] for _ in range(8)]
    all_large = [[] for _ in range(3)]
    labels = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Collecting feature maps"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            small_fmaps, large_fmaps = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                all_small[i].append(small_fmaps[i])
            for i in range(3):
                all_large[i].append(large_fmaps[i])

            labels.append(int(row["label"]))
        except Exception:
            continue

    return all_small, all_large, np.array(labels, dtype=np.int32)

def fit_all_spatial_pca_models(train_df, image_size, pixelhop_pca):
    all_small, all_large, labels = collect_feature_maps_for_training(
        train_df,
        image_size,
        pixelhop_pca,
    )

    small_models = []
    for i in range(8):
        small_models.append(fit_spatial_pca_for_blocktype(all_small[i]))

    large_models = []
    for i in range(3):
        large_models.append(fit_spatial_pca_for_blocktype(all_large[i]))

    return small_models, large_models

# ============================================================
# FEATURE MATRIX
# ============================================================

def build_feature_matrix(df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models):
    X_small = [[] for _ in range(8)]
    X_large = [[] for _ in range(3)]
    y = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Build features @ {image_size}"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            small_fmaps, large_fmaps = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                X_small[i].append(apply_spatial_pca_to_block(small_fmaps[i], small_spatial_models[i]))

            for i in range(3):
                X_large[i].append(apply_spatial_pca_to_block(large_fmaps[i], large_spatial_models[i]))

            y.append(int(row["label"]))
        except Exception:
            continue

    X_small = [np.stack(v, axis=0) for v in X_small]
    X_large = [np.stack(v, axis=0) for v in X_large]
    y = np.array(y, dtype=np.int32)

    return X_small, X_large, y

# ============================================================
# DFT FEATURE SELECTION
# ============================================================

def binary_cross_entropy_from_probs(y_true, p):
    eps = 1e-8
    p = np.clip(p, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)).mean()

def dft_score_feature(x, y):
    x = np.asarray(x).astype(np.float32)
    y = np.asarray(y).astype(np.int32)

    xmin, xmax = x.min(), x.max()
    if xmax <= xmin:
        return 1e9, None

    splits = np.linspace(xmin, xmax, cfg.dft_num_splits + 2)[1:-1]

    best_ce = 1e9
    best_split = None

    for s in splits:
        left = x <= s
        right = ~left

        if left.sum() == 0 or right.sum() == 0:
            continue

        p_left = y[left].mean()
        p_right = y[right].mean()
        probs = np.where(left, p_left, p_right)

        ce = binary_cross_entropy_from_probs(y, probs)

        if ce < best_ce:
            best_ce = ce
            best_split = s

    return best_ce, best_split

def fit_dft_selector(X, y, keep_ratio):
    scores = []
    splits = []

    for j in tqdm(range(X.shape[1]), desc="DFT"):
        ce, s = dft_score_feature(X[:, j], y)
        scores.append(ce)
        splits.append(s)

    scores = np.array(scores)
    order = np.argsort(scores)
    k = max(1, int(len(order) * keep_ratio))
    selected = order[:k]

    return {
        "selected_indices": selected,
        "scores": scores,
        "splits": splits,
    }

def apply_dft_selector(X, selector):
    return X[:, selector["selected_indices"]]

# ============================================================
# TRAIN PIPELINE
# ============================================================

def fit_defakehoppp_pipeline(train_df, val_df, image_size=224):
    t0 = time.time()

    print("\nFitting PixelHop PCA...")
    pixelhop_pca = fit_pixelhop_pca(train_df, image_size=image_size)
    t1 = time.time()

    print("\nFitting Spatial PCA...")
    small_spatial_models, large_spatial_models = fit_all_spatial_pca_models(
        train_df,
        image_size=image_size,
        pixelhop_pca=pixelhop_pca,
    )
    t2 = time.time()

    print("\nBuilding train and validation features...")
    Xs_train, Xl_train, y_train = build_feature_matrix(
        train_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )

    Xs_val, Xl_val, y_val = build_feature_matrix(
        val_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )
    t3 = time.time()

    print("\nRunning DFT feature selection...")
    small_selectors = []
    Xs_train_sel = []
    Xs_val_sel = []

    for i in range(8):
        sel = fit_dft_selector(Xs_train[i], y_train, cfg.landmark_keep_ratio)
        small_selectors.append(sel)
        Xs_train_sel.append(apply_dft_selector(Xs_train[i], sel))
        Xs_val_sel.append(apply_dft_selector(Xs_val[i], sel))

    large_selectors = []
    Xl_train_sel = []
    Xl_val_sel = []

    for i in range(3):
        sel = fit_dft_selector(Xl_train[i], y_train, cfg.region_keep_ratio)
        large_selectors.append(sel)
        Xl_train_sel.append(apply_dft_selector(Xl_train[i], sel))
        Xl_val_sel.append(apply_dft_selector(Xl_val[i], sel))

    X_train = np.concatenate(Xs_train_sel + Xl_train_sel, axis=1)
    X_val = np.concatenate(Xs_val_sel + Xl_val_sel, axis=1)
    t4 = time.time()

    print("\nTraining LightGBM classifier...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = lgb.LGBMClassifier(
        objective="binary",
        num_leaves=cfg.lgb_num_leaves,
        n_estimators=cfg.lgb_n_estimators,
        learning_rate=cfg.lgb_learning_rate,
        subsample=cfg.lgb_subsample,
        colsample_bytree=cfg.lgb_colsample_bytree,
        random_state=cfg.random_state,
    )

    clf.fit(X_train, y_train)
    t5 = time.time()

    timing = {
        "pixelhop_fit_sec": t1 - t0,
        "spatial_pca_fit_sec": t2 - t1,
        "feature_build_sec": t3 - t2,
        "dft_sec": t4 - t3,
        "classifier_sec": t5 - t4,
        "total_train_sec": t5 - t0,
    }

    model = {
        "pixelhop_pca": pixelhop_pca,
        "small_spatial_models": small_spatial_models,
        "large_spatial_models": large_spatial_models,
        "small_selectors": small_selectors,
        "large_selectors": large_selectors,
        "scaler": scaler,
        "clf": clf,
        "image_size": image_size,
    }

    return model, timing

# ============================================================
# EVALUATION
# ============================================================

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    uncertain_mask = (y_prob > cfg.uncertain_low) & (y_prob < cfg.uncertain_high)
    uncertain_rate = uncertain_mask.mean() * 100.0
    avg_uncertainty = (1.0 - 2.0 * np.abs(y_prob - 0.5)).mean()

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "avg_uncertainty": avg_uncertainty,
    }

def transform_dataframe_with_model(df, model, image_size=None):
    if image_size is None:
        image_size = model["image_size"]

    Xs, Xl, y = build_feature_matrix(
        df,
        image_size,
        model["pixelhop_pca"],
        model["small_spatial_models"],
        model["large_spatial_models"],
    )

    Xs_sel = [
        apply_dft_selector(Xs[i], model["small_selectors"][i])
        for i in range(8)
    ]

    Xl_sel = [
        apply_dft_selector(Xl[i], model["large_selectors"][i])
        for i in range(3)
    ]

    X = np.concatenate(Xs_sel + Xl_sel, axis=1)
    X = model["scaler"].transform(X)

    return X, y

def evaluate_model(model, df, image_size=None, model_name="DefakeHop++ Approx FF++ Balanced"):
    X, y = transform_dataframe_with_model(df, model, image_size=image_size)
    probs = model["clf"].predict_proba(X)[:, 1]

    m = compute_metrics(y, probs)

    return pd.DataFrame([{
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "AUC": m["auc"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Uncertain Rate (%)": m["uncertain_rate"],
        "Avg Uncertainty": m["avg_uncertainty"],
    }])

def evaluate_resolutions(model, test_df):
    rows = []

    for res in cfg.resolutions:
        X, y = transform_dataframe_with_model(test_df, model, image_size=res)
        probs = model["clf"].predict_proba(X)[:, 1]
        m = compute_metrics(y, probs)

        rows.append({
            "Resolution": res,
            "Accuracy": m["accuracy"],
            "AUC": m["auc"],
            "Fast (%)": 0.00,
            "Medium (%)": 0.00,
            "Full (%)": 100.00,
            "Avg Uncertainty": m["avg_uncertainty"],
            "Uncertain Rate (%)": m["uncertain_rate"],
        })

    return pd.DataFrame(rows)

def measure_inference_time(model, df, image_size, max_samples=500):
    sub_df = df.iloc[:max_samples].copy()

    start = time.time()
    X, y = transform_dataframe_with_model(sub_df, model, image_size=image_size)
    _ = model["clf"].predict_proba(X)[:, 1]
    elapsed = time.time() - start

    n = len(sub_df)
    return {
        "Mode": f"{image_size}x{image_size}",
        "Time per Image (ms)": (elapsed / max(n, 1)) * 1000.0,
        "FPS": n / max(elapsed, 1e-8),
    }

def measure_inference_time_by_resolution(model, test_df):
    rows = []
    for res in cfg.resolutions:
        rows.append(measure_inference_time(model, test_df, res, max_samples=500))
    return pd.DataFrame(rows)

def plot_resolution_results(res_df):
    tmp = res_df.sort_values("Resolution")

    plt.figure(figsize=(8, 4))
    plt.plot(tmp["Resolution"], tmp["Accuracy"], marker="o", label="Accuracy")
    plt.plot(tmp["Resolution"], tmp["AUC"], marker="s", label="AUC")
    plt.plot(tmp["Resolution"], tmp["Avg Uncertainty"], marker="^", label="Avg Uncertainty")
    plt.xlabel("Resolution")
    plt.ylabel("Score")
    plt.title("DefakeHop++ Approx FF++ Balanced Train vs Resolution")
    plt.grid(True)
    plt.legend()
    plt.show()

# ============================================================
# RUN
# ============================================================

print("\n" + "=" * 70)
print("RUNNING DEFAKEHOP++ APPROX ON FF++ WITH BALANCED TRAIN SET")
print("=" * 70)

model, timing = fit_defakehoppp_pipeline(
    train_df=train_df,
    val_df=val_df,
    image_size=cfg.image_size,
)

test_results_df = evaluate_model(
    model,
    test_df,
    image_size=cfg.image_size,
    model_name="DefakeHop++ Approx FF++ Balanced Train",
)

res_df = evaluate_resolutions(model, test_df)
inf_df = measure_inference_time_by_resolution(model, test_df)

train_time_df = pd.DataFrame([{
    "Model Type": "DefakeHop++ Approx FF++ Balanced Train",
    "mins": timing["total_train_sec"] / 60.0,
    "hrs": timing["total_train_sec"] / 3600.0,
}])

dataset_summary = pd.DataFrame([{
    "Experiment Name": "DefakeHop++ Approx Balanced Train",
    "Dataset": "FaceForensics++ C23",
    "Train Samples Original": len(train_df_full),
    "Train Samples Used Balanced": len(train_df),
    "Validation Samples": len(val_df),
    "Test Samples": len(test_df),
    "Train Real Used": int((train_df["label"] == 0).sum()),
    "Train Fake Used": int((train_df["label"] == 1).sum()),
    "Validation Real": int((val_df["label"] == 0).sum()),
    "Validation Fake": int((val_df["label"] == 1).sum()),
    "Test Real": int((test_df["label"] == 0).sum()),
    "Test Fake": int((test_df["label"] == 1).sum()),
    "Balancing Method": "Train-only undersampling of fake class to match real class",
    "Resolutions Used": str(cfg.resolutions),
    "Input Modality": "RGB handcrafted features",
    "Image Size": cfg.image_size,
}])

print("\nTest Results")
display(test_results_df)

print("\nAccuracy Based on Resolution")
display(res_df)

print("\nInference Time")
display(inf_df)

print("\nTraining Time")
display(train_time_df)

print("\nExperiment Summary")
display(dataset_summary)

plot_resolution_results(res_df)

# ============================================================
# SAVE
# ============================================================

test_results_df.to_csv(os.path.join(cfg.out_dir, "test_results.csv"), index=False)
res_df.to_csv(os.path.join(cfg.out_dir, "accuracy_based_on_resolution.csv"), index=False)
inf_df.to_csv(os.path.join(cfg.out_dir, "inference_time.csv"), index=False)
train_time_df.to_csv(os.path.join(cfg.out_dir, "training_time.csv"), index=False)
dataset_summary.to_csv(os.path.join(cfg.out_dir, "dataset_summary.csv"), index=False)
pd.DataFrame([timing]).to_csv(os.path.join(cfg.out_dir, "training_stage_timing_breakdown.csv"), index=False)

print("\nSaved outputs to:", cfg.out_dir)

In [ ]:
#FF++
!pip install lightgbm scikit-learn pillow tqdm matplotlib pandas numpy

import os
import io
import time
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

FRAMES_DIR = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames"
METADATA_CSV = "/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(42)

@dataclass
class DefakeHopPPConfig:
    image_size: int = 224

    small_block_size: int = 13
    large_block_size: int = 32
    patch_size: int = 3
    stride: int = 2

    max_train_fit_per_class: int = 1500
    dft_num_splits: int = 31

    landmark_keep_ratio: float = 0.35
    region_keep_ratio: float = 0.15

    spatial_pca_energy: float = 0.80
    spatial_pca_max_channels: int = 10

    uncertain_low: float = 0.4
    uncertain_high: float = 0.6

    lgb_num_leaves: int = 64
    lgb_n_estimators: int = 300
    lgb_learning_rate: float = 0.05
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8
    random_state: int = 42

    resolutions: tuple = (128, 224, 256, 384)
    out_dir: str = "./outputs_defakehoppp_ffpp"

cfg = DefakeHopPPConfig()
os.makedirs(cfg.out_dir, exist_ok=True)

# ============================================================
# FF++ DATASET LOADING
# ============================================================

def load_ffpp_metadata(metadata_csv):
    df = pd.read_csv(metadata_csv)

    required_cols = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)
    df = df.rename(columns={"frame_path": "image_path"})

    print("=" * 70)
    print("FF++ DATASET LOADED")
    print("=" * 70)
    print("Total usable frames:", len(df))

    for split in ["Train", "Val", "Test"]:
        split_df = df[df["split"].str.lower() == split.lower()]
        real = int((split_df["label"] == 0).sum())
        fake = int((split_df["label"] == 1).sum())
        videos = split_df["video_id"].nunique()

        print(
            f"{split}: real={real}, fake={fake}, "
            f"total={len(split_df)}, videos={videos}"
        )

    return df

df_meta = load_ffpp_metadata(METADATA_CSV)

train_df = df_meta[df_meta["split"].str.lower() == "train"].reset_index(drop=True)
val_df = df_meta[df_meta["split"].str.lower() == "val"].reset_index(drop=True)
test_df = df_meta[df_meta["split"].str.lower() == "test"].reset_index(drop=True)

print("\nTrain size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

# ============================================================
# IMAGE HELPERS
# ============================================================

def load_image_rgb(path):
    img = Image.open(path).convert("RGB")
    return np.array(img)

def resize_image_np(img, size):
    pil = Image.fromarray(img)
    pil = pil.resize((size, size))
    return np.array(pil)

def crop_square_block(img_rgb, center_xy, block_size):
    h, w = img_rgb.shape[:2]
    cx, cy = center_xy
    half = block_size // 2

    x1 = cx - half
    y1 = cy - half
    x2 = x1 + block_size
    y2 = y1 + block_size

    out = np.zeros((block_size, block_size, 3), dtype=np.uint8)

    src_x1 = max(0, x1)
    src_y1 = max(0, y1)
    src_x2 = min(w, x2)
    src_y2 = min(h, y2)

    dst_x1 = src_x1 - x1
    dst_y1 = src_y1 - y1
    dst_x2 = dst_x1 + (src_x2 - src_x1)
    dst_y2 = dst_y1 + (src_y2 - src_y1)

    out[dst_y1:dst_y2, dst_x1:dst_x2] = img_rgb[src_y1:src_y2, src_x1:src_x2]
    return out

# ============================================================
# DEFAKEHOP++ APPROX BLOCKS
# ============================================================

SMALL_BLOCK_CENTERS = [
    (0.32, 0.28),
    (0.45, 0.30),
    (0.68, 0.28),
    (0.55, 0.30),
    (0.38, 0.45),
    (0.62, 0.45),
    (0.50, 0.40),
    (0.50, 0.58),
]

LARGE_BLOCK_CENTERS = [
    (0.35, 0.35),
    (0.65, 0.35),
    (0.50, 0.68),
]

def extract_defakehoppp_blocks(img_rgb, small_block_size=13, large_block_size=32):
    h, w = img_rgb.shape[:2]

    small_blocks = []
    for nx, ny in SMALL_BLOCK_CENTERS:
        cx = int(nx * w)
        cy = int(ny * h)
        small_blocks.append(crop_square_block(img_rgb, (cx, cy), small_block_size))

    large_blocks = []
    for nx, ny in LARGE_BLOCK_CENTERS:
        cx = int(nx * w)
        cy = int(ny * h)
        large_blocks.append(crop_square_block(img_rgb, (cx, cy), large_block_size))

    return {
        "small_blocks": small_blocks,
        "large_blocks": large_blocks,
    }

def extract_patches_from_block(block, patch_size=3, stride=1):
    h, w, c = block.shape
    patches = []

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            p = block[y:y+patch_size, x:x+patch_size, :]
            patches.append(p.reshape(-1))

    return np.array(patches, dtype=np.float32)

# ============================================================
# PIXELHOP PCA
# ============================================================

def collect_training_patches(train_df, image_size=224, max_images_per_class=1500):
    real_paths = train_df[train_df["label"] == 0]["image_path"].tolist()[:max_images_per_class]
    fake_paths = train_df[train_df["label"] == 1]["image_path"].tolist()[:max_images_per_class]
    selected_paths = real_paths + fake_paths

    patches = []

    for path in tqdm(selected_paths, desc="Collecting training patches"):
        try:
            img = load_image_rgb(path)
            img = resize_image_np(img, image_size)

            blocks = extract_defakehoppp_blocks(
                img,
                small_block_size=cfg.small_block_size,
                large_block_size=cfg.large_block_size,
            )

            for b in blocks["small_blocks"] + blocks["large_blocks"]:
                p = extract_patches_from_block(
                    b,
                    patch_size=cfg.patch_size,
                    stride=1,
                )
                if len(p) > 0:
                    patches.append(p)

        except Exception:
            continue

    patches = np.concatenate(patches, axis=0)
    return patches

def fit_pixelhop_pca(train_df, image_size=224):
    patches = collect_training_patches(
        train_df,
        image_size=image_size,
        max_images_per_class=cfg.max_train_fit_per_class,
    )

    pca = PCA(n_components=27, random_state=cfg.random_state)
    pca.fit(patches)
    return pca

def pixelhop_transform_block(block, pca_model, patch_size=3, stride=2):
    h, w, c = block.shape
    out_h = (h - patch_size) // stride + 1
    out_w = (w - patch_size) // stride + 1

    feats = np.zeros((out_h, out_w, 27), dtype=np.float32)

    oy = 0
    for y in range(0, h - patch_size + 1, stride):
        ox = 0
        for x in range(0, w - patch_size + 1, stride):
            patch = block[y:y+patch_size, x:x+patch_size, :].reshape(1, -1).astype(np.float32)
            proj = pca_model.transform(patch)[0]
            feats[oy, ox, :] = proj
            ox += 1
        oy += 1

    return feats

def extract_all_fmaps_for_image(img_rgb, pixelhop_pca):
    blocks = extract_defakehoppp_blocks(
        img_rgb,
        small_block_size=cfg.small_block_size,
        large_block_size=cfg.large_block_size,
    )

    small_fmaps = [
        pixelhop_transform_block(
            b,
            pixelhop_pca,
            patch_size=cfg.patch_size,
            stride=cfg.stride,
        )
        for b in blocks["small_blocks"]
    ]

    large_fmaps = [
        pixelhop_transform_block(
            b,
            pixelhop_pca,
            patch_size=cfg.patch_size,
            stride=cfg.stride,
        )
        for b in blocks["large_blocks"]
    ]

    return {
        "small_fmaps": small_fmaps,
        "large_fmaps": large_fmaps,
    }

# ============================================================
# SPATIAL PCA
# ============================================================

def fit_spatial_pca_for_blocktype(feature_maps_list, energy_keep=0.80, max_channels=10):
    c = feature_maps_list[0].shape[-1]
    channel_models = {}
    kept_channels = []

    variances = []

    for ch in range(c):
        vals = []
        for fmap in feature_maps_list:
            vals.append(np.var(fmap[:, :, ch]))
        variances.append(np.mean(vals))

    ranked = np.argsort(variances)[::-1]
    kept = ranked[:max_channels]

    for ch in kept:
        X = []

        for fmap in feature_maps_list:
            X.append(fmap[:, :, ch].reshape(-1))

        X = np.stack(X, axis=0)

        pca = PCA(random_state=cfg.random_state)
        pca.fit(X)

        cum = np.cumsum(pca.explained_variance_ratio_)
        n_keep = int(np.searchsorted(cum, energy_keep) + 1)
        n_keep = max(1, n_keep)

        channel_models[ch] = {
            "mean": pca.mean_.copy(),
            "components": pca.components_[:n_keep].copy(),
        }

        kept_channels.append(ch)

    return {
        "kept_channels": kept_channels,
        "channel_models": channel_models,
    }

def apply_spatial_pca_to_block(fmap, spatial_pca_model):
    feats = []

    for ch in spatial_pca_model["kept_channels"]:
        vec = fmap[:, :, ch].reshape(1, -1)
        mean = spatial_pca_model["channel_models"][ch]["mean"]
        comps = spatial_pca_model["channel_models"][ch]["components"]
        proj = (vec - mean) @ comps.T
        feats.append(proj[0])

    if len(feats) == 0:
        return np.zeros((1,), dtype=np.float32)

    return np.concatenate(feats, axis=0).astype(np.float32)

def collect_feature_maps_for_training(df, image_size, pixelhop_pca, limit=None):
    rows = df if limit is None else df.iloc[:limit]

    all_small = [[] for _ in range(8)]
    all_large = [[] for _ in range(3)]
    labels = []

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc="Collecting feature maps"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            out = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                all_small[i].append(out["small_fmaps"][i])

            for i in range(3):
                all_large[i].append(out["large_fmaps"][i])

            labels.append(int(row["label"]))

        except Exception:
            continue

    return all_small, all_large, np.array(labels, dtype=np.int32)

def fit_all_spatial_pca_models(train_df, image_size, pixelhop_pca, limit=None):
    all_small, all_large, labels = collect_feature_maps_for_training(
        train_df,
        image_size,
        pixelhop_pca,
        limit=limit,
    )

    small_models = []
    for i in range(8):
        m = fit_spatial_pca_for_blocktype(
            all_small[i],
            energy_keep=cfg.spatial_pca_energy,
            max_channels=cfg.spatial_pca_max_channels,
        )
        small_models.append(m)

    large_models = []
    for i in range(3):
        m = fit_spatial_pca_for_blocktype(
            all_large[i],
            energy_keep=cfg.spatial_pca_energy,
            max_channels=cfg.spatial_pca_max_channels,
        )
        large_models.append(m)

    return small_models, large_models

# ============================================================
# FEATURE MATRIX
# ============================================================

def build_feature_matrix(df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models):
    X_small = [[] for _ in range(8)]
    X_large = [[] for _ in range(3)]
    y = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Build features @ {image_size}"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            out = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                feat = apply_spatial_pca_to_block(
                    out["small_fmaps"][i],
                    small_spatial_models[i],
                )
                X_small[i].append(feat)

            for i in range(3):
                feat = apply_spatial_pca_to_block(
                    out["large_fmaps"][i],
                    large_spatial_models[i],
                )
                X_large[i].append(feat)

            y.append(int(row["label"]))

        except Exception:
            continue

    X_small = [np.stack(v, axis=0) for v in X_small]
    X_large = [np.stack(v, axis=0) for v in X_large]
    y = np.array(y, dtype=np.int32)

    return X_small, X_large, y

# ============================================================
# DFT FEATURE SELECTION
# ============================================================

def binary_cross_entropy_from_probs(y_true, p):
    eps = 1e-8
    p = np.clip(p, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)).mean()

def dft_score_feature(x, y, num_splits=31):
    x = np.asarray(x).astype(np.float32)
    y = np.asarray(y).astype(np.int32)

    xmin, xmax = x.min(), x.max()

    if xmax <= xmin:
        return 1e9, None

    splits = np.linspace(xmin, xmax, num_splits + 2)[1:-1]

    best_ce = 1e9
    best_split = None

    for s in splits:
        left = x <= s
        right = ~left

        if left.sum() == 0 or right.sum() == 0:
            continue

        p_left = y[left].mean()
        p_right = y[right].mean()

        probs = np.where(left, p_left, p_right)
        ce = binary_cross_entropy_from_probs(y, probs)

        if ce < best_ce:
            best_ce = ce
            best_split = s

    return best_ce, best_split

def fit_dft_selector(X, y, keep_ratio=0.35, num_splits=31):
    scores = []
    splits = []

    for j in tqdm(range(X.shape[1]), desc="DFT"):
        ce, s = dft_score_feature(X[:, j], y, num_splits=num_splits)
        scores.append(ce)
        splits.append(s)

    scores = np.array(scores)
    order = np.argsort(scores)
    k = max(1, int(len(order) * keep_ratio))
    selected = order[:k]

    return {
        "selected_indices": selected,
        "scores": scores,
        "splits": splits,
    }

def apply_dft_selector(X, selector):
    return X[:, selector["selected_indices"]]

# ============================================================
# TRAIN PIPELINE
# ============================================================

def fit_defakehoppp_pipeline(train_df, val_df, image_size=224):
    t0 = time.time()

    print("\nFitting PixelHop PCA...")
    pixelhop_pca = fit_pixelhop_pca(train_df, image_size=image_size)
    t1 = time.time()

    print("\nFitting Spatial PCA...")
    small_spatial_models, large_spatial_models = fit_all_spatial_pca_models(
        train_df,
        image_size=image_size,
        pixelhop_pca=pixelhop_pca,
    )
    t2 = time.time()

    print("\nBuilding train and validation features...")
    Xs_train, Xl_train, y_train = build_feature_matrix(
        train_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )

    Xs_val, Xl_val, y_val = build_feature_matrix(
        val_df,
        image_size,
        pixelhop_pca,
        small_spatial_models,
        large_spatial_models,
    )
    t3 = time.time()

    print("\nRunning DFT feature selection...")
    small_selectors = []
    Xs_train_sel = []
    Xs_val_sel = []

    for i in range(8):
        sel = fit_dft_selector(
            Xs_train[i],
            y_train,
            keep_ratio=cfg.landmark_keep_ratio,
            num_splits=cfg.dft_num_splits,
        )
        small_selectors.append(sel)
        Xs_train_sel.append(apply_dft_selector(Xs_train[i], sel))
        Xs_val_sel.append(apply_dft_selector(Xs_val[i], sel))

    large_selectors = []
    Xl_train_sel = []
    Xl_val_sel = []

    for i in range(3):
        sel = fit_dft_selector(
            Xl_train[i],
            y_train,
            keep_ratio=cfg.region_keep_ratio,
            num_splits=cfg.dft_num_splits,
        )
        large_selectors.append(sel)
        Xl_train_sel.append(apply_dft_selector(Xl_train[i], sel))
        Xl_val_sel.append(apply_dft_selector(Xl_val[i], sel))

    X_train = np.concatenate(Xs_train_sel + Xl_train_sel, axis=1)
    X_val = np.concatenate(Xs_val_sel + Xl_val_sel, axis=1)
    t4 = time.time()

    print("\nTraining LightGBM classifier...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = lgb.LGBMClassifier(
        objective="binary",
        num_leaves=cfg.lgb_num_leaves,
        n_estimators=cfg.lgb_n_estimators,
        learning_rate=cfg.lgb_learning_rate,
        subsample=cfg.lgb_subsample,
        colsample_bytree=cfg.lgb_colsample_bytree,
        random_state=cfg.random_state,
    )

    clf.fit(X_train, y_train)
    t5 = time.time()

    timing = {
        "pixelhop_fit_sec": t1 - t0,
        "spatial_pca_fit_sec": t2 - t1,
        "feature_build_sec": t3 - t2,
        "dft_sec": t4 - t3,
        "classifier_sec": t5 - t4,
        "total_train_sec": t5 - t0,
    }

    model = {
        "pixelhop_pca": pixelhop_pca,
        "small_spatial_models": small_spatial_models,
        "large_spatial_models": large_spatial_models,
        "small_selectors": small_selectors,
        "large_selectors": large_selectors,
        "scaler": scaler,
        "clf": clf,
        "image_size": image_size,
    }

    return model, timing

# ============================================================
# EVALUATION
# ============================================================

def compute_metrics(y_true, y_prob, uncertain_low=0.4, uncertain_high=0.6):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    uncertain_mask = (y_prob > uncertain_low) & (y_prob < uncertain_high)
    uncertain_rate = uncertain_mask.mean() * 100.0
    avg_uncertainty = (1.0 - 2.0 * np.abs(y_prob - 0.5)).mean()

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "avg_uncertainty": avg_uncertainty,
    }

def transform_dataframe_with_model(df, model, image_size=None):
    if image_size is None:
        image_size = model["image_size"]

    Xs, Xl, y = build_feature_matrix(
        df,
        image_size,
        model["pixelhop_pca"],
        model["small_spatial_models"],
        model["large_spatial_models"],
    )

    Xs_sel = [
        apply_dft_selector(Xs[i], model["small_selectors"][i])
        for i in range(8)
    ]

    Xl_sel = [
        apply_dft_selector(Xl[i], model["large_selectors"][i])
        for i in range(3)
    ]

    X = np.concatenate(Xs_sel + Xl_sel, axis=1)
    X = model["scaler"].transform(X)

    return X, y

def evaluate_model(model, df, image_size=None, model_name="DefakeHop++ Approx FF++"):
    X, y = transform_dataframe_with_model(df, model, image_size=image_size)
    probs = model["clf"].predict_proba(X)[:, 1]

    m = compute_metrics(
        y,
        probs,
        uncertain_low=cfg.uncertain_low,
        uncertain_high=cfg.uncertain_high,
    )

    return pd.DataFrame([{
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "AUC": m["auc"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Uncertain Rate (%)": m["uncertain_rate"],
        "Avg Uncertainty": m["avg_uncertainty"],
    }])

def evaluate_resolutions(model, test_df, resolutions=(128, 224, 256, 384)):
    rows = []

    for res in resolutions:
        X, y = transform_dataframe_with_model(test_df, model, image_size=res)
        probs = model["clf"].predict_proba(X)[:, 1]

        m = compute_metrics(
            y,
            probs,
            uncertain_low=cfg.uncertain_low,
            uncertain_high=cfg.uncertain_high,
        )

        rows.append({
            "Resolution": res,
            "Accuracy": m["accuracy"],
            "AUC": m["auc"],
            "Fast (%)": 0.00,
            "Medium (%)": 0.00,
            "Full (%)": 100.00,
            "Avg Uncertainty": m["avg_uncertainty"],
            "Uncertain Rate (%)": m["uncertain_rate"],
        })

    return pd.DataFrame(rows)

def measure_inference_time(model, df, image_size, max_samples=500):
    sub_df = df.iloc[:max_samples].copy()

    start = time.time()
    X, y = transform_dataframe_with_model(sub_df, model, image_size=image_size)
    _ = model["clf"].predict_proba(X)[:, 1]
    elapsed = time.time() - start

    n = len(sub_df)
    ms_per_image = (elapsed / max(n, 1)) * 1000.0
    fps = n / max(elapsed, 1e-8)

    return {
        "Mode": f"{image_size}x{image_size}",
        "Time per Image (ms)": ms_per_image,
        "FPS": fps,
    }

def measure_inference_time_by_resolution(model, test_df, resolutions=(128, 224, 256, 384), max_samples=500):
    rows = []

    for res in resolutions:
        rows.append(
            measure_inference_time(
                model,
                test_df,
                image_size=res,
                max_samples=max_samples,
            )
        )

    return pd.DataFrame(rows)

def make_training_time_table(timing_dict):
    total_mins = timing_dict["total_train_sec"] / 60.0

    return pd.DataFrame([{
        "Model Type": "DefakeHop++ Approx FF++",
        "mins": total_mins,
        "hrs": total_mins / 60.0,
    }])

def plot_resolution_results(res_df):
    tmp = res_df.copy()
    tmp = tmp.sort_values("Resolution")

    plt.figure(figsize=(8, 4))
    plt.plot(tmp["Resolution"], tmp["Accuracy"], marker="o", label="Accuracy")
    plt.plot(tmp["Resolution"], tmp["AUC"], marker="s", label="AUC")
    plt.plot(tmp["Resolution"], tmp["Avg Uncertainty"], marker="^", label="Avg Uncertainty")
    plt.xlabel("Resolution")
    plt.ylabel("Score")
    plt.title("DefakeHop++ Approx FF++ vs Resolution")
    plt.grid(True)
    plt.legend()
    plt.show()

# ============================================================
# RUN FULL PIPELINE
# ============================================================

print("\n" + "=" * 70)
print("RUNNING DEFAKEHOP++ APPROX ON FF++")
print("=" * 70)

model, timing = fit_defakehoppp_pipeline(
    train_df=train_df,
    val_df=val_df,
    image_size=cfg.image_size,
)

print("\nTraining timing:")
print(timing)

test_results_df = evaluate_model(
    model,
    test_df,
    image_size=cfg.image_size,
    model_name="DefakeHop++ Approx FF++",
)

res_df = evaluate_resolutions(
    model,
    test_df,
    resolutions=cfg.resolutions,
)

inf_df = measure_inference_time_by_resolution(
    model,
    test_df,
    resolutions=cfg.resolutions,
    max_samples=500,
)

train_time_df = make_training_time_table(timing)

print("\nTest Results")
display(test_results_df)

print("\nAccuracy Based on Resolution")
display(res_df)

print("\nInference Time")
display(inf_df)

print("\nTraining Time")
display(train_time_df)

plot_resolution_results(res_df)

# ============================================================
# SAVE OUTPUTS
# ============================================================

test_results_df.to_csv(os.path.join(cfg.out_dir, "test_results.csv"), index=False)
res_df.to_csv(os.path.join(cfg.out_dir, "accuracy_based_on_resolution.csv"), index=False)
inf_df.to_csv(os.path.join(cfg.out_dir, "inference_time.csv"), index=False)
train_time_df.to_csv(os.path.join(cfg.out_dir, "training_time.csv"), index=False)
pd.DataFrame([timing]).to_csv(
    os.path.join(cfg.out_dir, "training_stage_timing_breakdown.csv"),
    index=False,
)

dataset_summary = pd.DataFrame([{
    "Experiment Name": "DefakeHop++ Approx FF++",
    "Dataset": "FaceForensics++ C23",
    "Train Samples": len(train_df),
    "Validation Samples": len(val_df),
    "Test Samples": len(test_df),
    "Train Real": int((train_df["label"] == 0).sum()),
    "Train Fake": int((train_df["label"] == 1).sum()),
    "Validation Real": int((val_df["label"] == 0).sum()),
    "Validation Fake": int((val_df["label"] == 1).sum()),
    "Test Real": int((test_df["label"] == 0).sum()),
    "Test Fake": int((test_df["label"] == 1).sum()),
    "Resolutions Used": str(cfg.resolutions),
    "Input Modality": "RGB",
    "Image Size": cfg.image_size,
}])

dataset_summary.to_csv(os.path.join(cfg.out_dir, "dataset_summary.csv"), index=False)

print("\nSaved outputs to:", cfg.out_dir)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pranabr0y/celebdf-v2image-dataset")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
import pandas as pd

root = Path(path) / "Celeb_V2"
print(root)

In [ ]:
def build_csv_from_split(split_path):
    rows = []

    real_dir = split_path / "real"
    fake_dir = split_path / "fake"

    for img_path in real_dir.glob("*"):
        if img_path.is_file():
            rows.append({
                "image_path": str(img_path),
                "label": 0
            })

    for img_path in fake_dir.glob("*"):
        if img_path.is_file():
            rows.append({
                "image_path": str(img_path),
                "label": 1
            })

    return pd.DataFrame(rows)

In [ ]:
train_df = build_csv_from_split(root / "Train")
val_df = build_csv_from_split(root / "Val")
test_df = build_csv_from_split(root / "Test")

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

In [ ]:
import os
import time
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(42)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(42)


@dataclass
class DefakeHopPPConfig:
    image_size: int = 224

    small_block_size: int = 13
    large_block_size: int = 32
    patch_size: int = 3
    stride: int = 2

    max_train_fit_per_class: int = 1500
    dft_num_splits: int = 31

    landmark_keep_ratio: float = 0.35
    region_keep_ratio: float = 0.15

    spatial_pca_energy: float = 0.80
    spatial_pca_max_channels: int = 10

    uncertain_low: float = 0.4
    uncertain_high: float = 0.6

    lgb_num_leaves: int = 64
    lgb_n_estimators: int = 300
    lgb_learning_rate: float = 0.05
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8
    random_state: int = 42

    resolutions: tuple = (128, 224, 256, 384)

    out_dir: str = "./outputs_defakehoppp"

cfg = DefakeHopPPConfig()
os.makedirs(cfg.out_dir, exist_ok=True)

In [ ]:
def load_image_rgb(path):
    img = Image.open(path).convert("RGB")
    return np.array(img)

def resize_image_np(img, size):
    pil = Image.fromarray(img)
    pil = pil.resize((size, size))
    return np.array(pil)

def crop_square_block(img_rgb, center_xy, block_size):
    h, w = img_rgb.shape[:2]
    cx, cy = center_xy
    half = block_size // 2

    x1 = cx - half
    y1 = cy - half
    x2 = x1 + block_size
    y2 = y1 + block_size

    out = np.zeros((block_size, block_size, 3), dtype=np.uint8)

    src_x1 = max(0, x1)
    src_y1 = max(0, y1)
    src_x2 = min(w, x2)
    src_y2 = min(h, y2)

    dst_x1 = src_x1 - x1
    dst_y1 = src_y1 - y1
    dst_x2 = dst_x1 + (src_x2 - src_x1)
    dst_y2 = dst_y1 + (src_y2 - src_y1)

    out[dst_y1:dst_y2, dst_x1:dst_x2] = img_rgb[src_y1:src_y2, src_x1:src_x2]
    return out


In [ ]:
SMALL_BLOCK_CENTERS = [
    (0.32, 0.28),
    (0.45, 0.30),
    (0.68, 0.28),
    (0.55, 0.30),
    (0.38, 0.45),
    (0.62, 0.45),
    (0.50, 0.40),
    (0.50, 0.58),
]

LARGE_BLOCK_CENTERS = [
    (0.35, 0.35),   # left eye region
    (0.65, 0.35),   # right eye region
    (0.50, 0.68),   # mouth region
]

def extract_defakehoppp_blocks(img_rgb, small_block_size=13, large_block_size=32):
    h, w = img_rgb.shape[:2]

    small_blocks = []
    for nx, ny in SMALL_BLOCK_CENTERS:
        cx = int(nx * w)
        cy = int(ny * h)
        block = crop_square_block(img_rgb, (cx, cy), small_block_size)
        small_blocks.append(block)

    large_blocks = []
    for nx, ny in LARGE_BLOCK_CENTERS:
        cx = int(nx * w)
        cy = int(ny * h)
        block = crop_square_block(img_rgb, (cx, cy), large_block_size)
        large_blocks.append(block)

    return {
        "small_blocks": small_blocks,
        "large_blocks": large_blocks
    }


In [ ]:
def extract_patches_from_block(block, patch_size=3, stride=1):
    h, w, c = block.shape
    patches = []
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            p = block[y:y+patch_size, x:x+patch_size, :]
            patches.append(p.reshape(-1))
    return np.array(patches, dtype=np.float32)



In [ ]:
def collect_training_patches(train_df, image_size=224, max_images_per_class=1500):
    real_paths = train_df[train_df["label"] == 0]["image_path"].tolist()[:max_images_per_class]
    fake_paths = train_df[train_df["label"] == 1]["image_path"].tolist()[:max_images_per_class]
    selected_paths = real_paths + fake_paths

    patches = []
    for path in tqdm(selected_paths, desc="Collecting training patches"):
        try:
            img = load_image_rgb(path)
            img = resize_image_np(img, image_size)
            blocks = extract_defakehoppp_blocks(
                img,
                small_block_size=cfg.small_block_size,
                large_block_size=cfg.large_block_size
            )

            for b in blocks["small_blocks"] + blocks["large_blocks"]:
                p = extract_patches_from_block(b, patch_size=cfg.patch_size, stride=1)
                if len(p) > 0:
                    patches.append(p)
        except:
            continue

    patches = np.concatenate(patches, axis=0)
    return patches


def fit_pixelhop_pca(train_df, image_size=224):
    patches = collect_training_patches(
        train_df,
        image_size=image_size,
        max_images_per_class=cfg.max_train_fit_per_class
    )

    pca = PCA(n_components=27, random_state=cfg.random_state)
    pca.fit(patches)
    return pca


In [ ]:
def collect_training_patches(train_df, image_size=224, max_images_per_class=1500):
    real_paths = train_df[train_df["label"] == 0]["image_path"].tolist()[:max_images_per_class]
    fake_paths = train_df[train_df["label"] == 1]["image_path"].tolist()[:max_images_per_class]
    selected_paths = real_paths + fake_paths

    patches = []
    for path in tqdm(selected_paths, desc="Collecting training patches"):
        try:
            img = load_image_rgb(path)
            img = resize_image_np(img, image_size)
            blocks = extract_defakehoppp_blocks(
                img,
                small_block_size=cfg.small_block_size,
                large_block_size=cfg.large_block_size
            )

            for b in blocks["small_blocks"] + blocks["large_blocks"]:
                p = extract_patches_from_block(b, patch_size=cfg.patch_size, stride=1)
                if len(p) > 0:
                    patches.append(p)
        except:
            continue

    patches = np.concatenate(patches, axis=0)
    return patches


def fit_pixelhop_pca(train_df, image_size=224):
    patches = collect_training_patches(
        train_df,
        image_size=image_size,
        max_images_per_class=cfg.max_train_fit_per_class
    )

    pca = PCA(n_components=27, random_state=cfg.random_state)
    pca.fit(patches)
    return pca


# =========================================
# 6. PIXELHOP TRANSFORM
# =========================================
def pixelhop_transform_block(block, pca_model, patch_size=3, stride=2):
    h, w, c = block.shape
    out_h = (h - patch_size) // stride + 1
    out_w = (w - patch_size) // stride + 1

    feats = np.zeros((out_h, out_w, 27), dtype=np.float32)

    oy = 0
    for y in range(0, h - patch_size + 1, stride):
        ox = 0
        for x in range(0, w - patch_size + 1, stride):
            patch = block[y:y+patch_size, x:x+patch_size, :].reshape(1, -1).astype(np.float32)
            proj = pca_model.transform(patch)[0]
            feats[oy, ox, :] = proj
            ox += 1
        oy += 1

    return feats


def extract_all_fmaps_for_image(img_rgb, pixelhop_pca):
    blocks = extract_defakehoppp_blocks(
        img_rgb,
        small_block_size=cfg.small_block_size,
        large_block_size=cfg.large_block_size
    )

    small_fmaps = [
        pixelhop_transform_block(b, pixelhop_pca, patch_size=cfg.patch_size, stride=cfg.stride)
        for b in blocks["small_blocks"]
    ]
    large_fmaps = [
        pixelhop_transform_block(b, pixelhop_pca, patch_size=cfg.patch_size, stride=cfg.stride)
        for b in blocks["large_blocks"]
    ]

    return {
        "small_fmaps": small_fmaps,
        "large_fmaps": large_fmaps
    }



In [ ]:
def fit_spatial_pca_for_blocktype(feature_maps_list, energy_keep=0.80, max_channels=10):
    c = feature_maps_list[0].shape[-1]
    channel_models = {}
    kept_channels = []

    variances = []
    for ch in range(c):
        vals = []
        for fmap in feature_maps_list:
            vals.append(np.var(fmap[:, :, ch]))
        variances.append(np.mean(vals))

    ranked = np.argsort(variances)[::-1]
    kept = ranked[:max_channels]

    for ch in kept:
        X = []
        for fmap in feature_maps_list:
            X.append(fmap[:, :, ch].reshape(-1))
        X = np.stack(X, axis=0)

        pca = PCA(random_state=cfg.random_state)
        pca.fit(X)

        cum = np.cumsum(pca.explained_variance_ratio_)
        n_keep = int(np.searchsorted(cum, energy_keep) + 1)
        n_keep = max(1, n_keep)

        channel_models[ch] = {
            "mean": pca.mean_.copy(),
            "components": pca.components_[:n_keep].copy()
        }
        kept_channels.append(ch)

    return {
        "kept_channels": kept_channels,
        "channel_models": channel_models
    }


def apply_spatial_pca_to_block(fmap, spatial_pca_model):
    feats = []
    for ch in spatial_pca_model["kept_channels"]:
        vec = fmap[:, :, ch].reshape(1, -1)
        mean = spatial_pca_model["channel_models"][ch]["mean"]
        comps = spatial_pca_model["channel_models"][ch]["components"]
        proj = (vec - mean) @ comps.T
        feats.append(proj[0])
    if len(feats) == 0:
        return np.zeros((1,), dtype=np.float32)
    return np.concatenate(feats, axis=0).astype(np.float32)


In [ ]:
def collect_feature_maps_for_training(df, image_size, pixelhop_pca, limit=None):
    rows = df if limit is None else df.iloc[:limit]
    all_small = [[] for _ in range(8)]
    all_large = [[] for _ in range(3)]
    labels = []

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc="Collecting feature maps"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            out = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                all_small[i].append(out["small_fmaps"][i])
            for i in range(3):
                all_large[i].append(out["large_fmaps"][i])

            labels.append(int(row["label"]))
        except:
            continue

    return all_small, all_large, np.array(labels, dtype=np.int32)


def fit_all_spatial_pca_models(train_df, image_size, pixelhop_pca, limit=None):
    all_small, all_large, labels = collect_feature_maps_for_training(
        train_df, image_size, pixelhop_pca, limit=limit
    )

    small_models = []
    for i in range(8):
        m = fit_spatial_pca_for_blocktype(
            all_small[i],
            energy_keep=cfg.spatial_pca_energy,
            max_channels=cfg.spatial_pca_max_channels
        )
        small_models.append(m)

    large_models = []
    for i in range(3):
        m = fit_spatial_pca_for_blocktype(
            all_large[i],
            energy_keep=cfg.spatial_pca_energy,
            max_channels=cfg.spatial_pca_max_channels
        )
        large_models.append(m)

    return small_models, large_models


In [ ]:
def build_feature_matrix(df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models):
    X_small = [[] for _ in range(8)]
    X_large = [[] for _ in range(3)]
    y = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Build features @ {image_size}"):
        try:
            img = load_image_rgb(row["image_path"])
            img = resize_image_np(img, image_size)
            out = extract_all_fmaps_for_image(img, pixelhop_pca)

            for i in range(8):
                feat = apply_spatial_pca_to_block(out["small_fmaps"][i], small_spatial_models[i])
                X_small[i].append(feat)

            for i in range(3):
                feat = apply_spatial_pca_to_block(out["large_fmaps"][i], large_spatial_models[i])
                X_large[i].append(feat)

            y.append(int(row["label"]))
        except:
            continue

    X_small = [np.stack(v, axis=0) for v in X_small]
    X_large = [np.stack(v, axis=0) for v in X_large]
    y = np.array(y, dtype=np.int32)

    return X_small, X_large, y

In [ ]:
def binary_cross_entropy_from_probs(y_true, p):
    eps = 1e-8
    p = np.clip(p, eps, 1 - eps)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)).mean()

def dft_score_feature(x, y, num_splits=31):
    x = np.asarray(x).astype(np.float32)
    y = np.asarray(y).astype(np.int32)

    xmin, xmax = x.min(), x.max()
    if xmax <= xmin:
        return 1e9, None

    splits = np.linspace(xmin, xmax, num_splits + 2)[1:-1]

    best_ce = 1e9
    best_split = None

    for s in splits:
        left = x <= s
        right = ~left

        if left.sum() == 0 or right.sum() == 0:
            continue

        p_left = y[left].mean()
        p_right = y[right].mean()

        probs = np.where(left, p_left, p_right)
        ce = binary_cross_entropy_from_probs(y, probs)

        if ce < best_ce:
            best_ce = ce
            best_split = s

    return best_ce, best_split

def fit_dft_selector(X, y, keep_ratio=0.35, num_splits=31):
    scores = []
    splits = []

    for j in tqdm(range(X.shape[1]), desc="DFT"):
        ce, s = dft_score_feature(X[:, j], y, num_splits=num_splits)
        scores.append(ce)
        splits.append(s)

    scores = np.array(scores)
    order = np.argsort(scores)
    k = max(1, int(len(order) * keep_ratio))
    selected = order[:k]

    return {
        "selected_indices": selected,
        "scores": scores,
        "splits": splits
    }

def apply_dft_selector(X, selector):
    return X[:, selector["selected_indices"]]


In [ ]:
def fit_defakehoppp_pipeline(train_df, val_df, image_size=224):
    start_total = time.time()

    t0 = time.time()
    pixelhop_pca = fit_pixelhop_pca(train_df, image_size=image_size)
    t1 = time.time()

    small_spatial_models, large_spatial_models = fit_all_spatial_pca_models(
        train_df, image_size=image_size, pixelhop_pca=pixelhop_pca
    )
    t2 = time.time()

    Xs_train, Xl_train, y_train = build_feature_matrix(
        train_df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models
    )
    Xs_val, Xl_val, y_val = build_feature_matrix(
        val_df, image_size, pixelhop_pca, small_spatial_models, large_spatial_models
    )
    t3 = time.time()

    small_selectors = []
    Xs_train_sel = []
    Xs_val_sel = []

    for i in range(8):
        sel = fit_dft_selector(
            Xs_train[i], y_train,
            keep_ratio=cfg.landmark_keep_ratio,
            num_splits=cfg.dft_num_splits
        )
        small_selectors.append(sel)
        Xs_train_sel.append(apply_dft_selector(Xs_train[i], sel))
        Xs_val_sel.append(apply_dft_selector(Xs_val[i], sel))

    large_selectors = []
    Xl_train_sel = []
    Xl_val_sel = []

    for i in range(3):
        sel = fit_dft_selector(
            Xl_train[i], y_train,
            keep_ratio=cfg.region_keep_ratio,
            num_splits=cfg.dft_num_splits
        )
        large_selectors.append(sel)
        Xl_train_sel.append(apply_dft_selector(Xl_train[i], sel))
        Xl_val_sel.append(apply_dft_selector(Xl_val[i], sel))

    X_train = np.concatenate(Xs_train_sel + Xl_train_sel, axis=1)
    X_val = np.concatenate(Xs_val_sel + Xl_val_sel, axis=1)
    t4 = time.time()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    clf = lgb.LGBMClassifier(
        objective="binary",
        num_leaves=cfg.lgb_num_leaves,
        n_estimators=cfg.lgb_n_estimators,
        learning_rate=cfg.lgb_learning_rate,
        subsample=cfg.lgb_subsample,
        colsample_bytree=cfg.lgb_colsample_bytree,
        random_state=cfg.random_state
    )
    clf.fit(X_train, y_train)
    t5 = time.time()

    timing = {
        "pixelhop_fit_sec": t1 - t0,
        "spatial_pca_fit_sec": t2 - t1,
        "feature_build_sec": t3 - t2,
        "dft_sec": t4 - t3,
        "classifier_sec": t5 - t4,
        "total_train_sec": t5 - t0
    }

    model = {
        "pixelhop_pca": pixelhop_pca,
        "small_spatial_models": small_spatial_models,
        "large_spatial_models": large_spatial_models,
        "small_selectors": small_selectors,
        "large_selectors": large_selectors,
        "scaler": scaler,
        "clf": clf,
        "image_size": image_size
    }

    return model, timing

In [ ]:
def compute_metrics(y_true, y_prob, uncertain_low=0.4, uncertain_high=0.6):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except:
        auc = np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    uncertain_mask = (y_prob > uncertain_low) & (y_prob < uncertain_high)
    uncertain_rate = uncertain_mask.mean() * 100.0
    avg_uncertainty = (1.0 - 2.0 * np.abs(y_prob - 0.5)).mean()

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "avg_uncertainty": avg_uncertainty
    }


In [ ]:
def transform_dataframe_with_model(df, model, image_size=None):
    if image_size is None:
        image_size = model["image_size"]

    Xs, Xl, y = build_feature_matrix(
        df,
        image_size,
        model["pixelhop_pca"],
        model["small_spatial_models"],
        model["large_spatial_models"]
    )

    Xs_sel = [apply_dft_selector(Xs[i], model["small_selectors"][i]) for i in range(8)]
    Xl_sel = [apply_dft_selector(Xl[i], model["large_selectors"][i]) for i in range(3)]

    X = np.concatenate(Xs_sel + Xl_sel, axis=1)
    X = model["scaler"].transform(X)
    return X, y


def evaluate_model(model, df, image_size=None, model_name="DefakeHop++ Approx"):
    X, y = transform_dataframe_with_model(df, model, image_size=image_size)
    probs = model["clf"].predict_proba(X)[:, 1]

    m = compute_metrics(
        y, probs,
        uncertain_low=cfg.uncertain_low,
        uncertain_high=cfg.uncertain_high
    )

    return pd.DataFrame([{
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "AUC": m["auc"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "Uncertain Rate (%)": m["uncertain_rate"],
        "Avg Uncertainty": m["avg_uncertainty"]
    }])


def evaluate_resolutions(model, test_df, resolutions=(128, 224, 256, 384)):
    rows = []
    for res in resolutions:
        X, y = transform_dataframe_with_model(test_df, model, image_size=res)
        probs = model["clf"].predict_proba(X)[:, 1]

        m = compute_metrics(
            y, probs,
            uncertain_low=cfg.uncertain_low,
            uncertain_high=cfg.uncertain_high
        )

        rows.append({
            "Resolution": f"{res}x{res}",
            "Accuracy": m["accuracy"],
            "AUC": m["auc"],
            "Avg Uncertainty": m["avg_uncertainty"],
            "Uncertain Rate (%)": m["uncertain_rate"]
        })
    return pd.DataFrame(rows)


In [ ]:
def measure_inference_time(model, df, image_size, max_samples=500):
    sub_df = df.iloc[:max_samples].copy()

    start = time.time()
    X, y = transform_dataframe_with_model(sub_df, model, image_size=image_size)
    _ = model["clf"].predict_proba(X)[:, 1]
    elapsed = time.time() - start

    n = len(sub_df)
    ms_per_image = (elapsed / max(n, 1)) * 1000.0
    fps = n / max(elapsed, 1e-8)

    return {
        "Mode": f"{image_size}x{image_size}",
        "Time per Image (ms)": ms_per_image,
        "FPS": fps
    }


def measure_inference_time_by_resolution(model, test_df, resolutions=(128, 224, 256, 384), max_samples=500):
    rows = []
    for res in resolutions:
        rows.append(measure_inference_time(model, test_df, image_size=res, max_samples=max_samples))
    return pd.DataFrame(rows)


def make_training_time_table(timing_dict):
    total_mins = timing_dict["total_train_sec"] / 60.0
    return pd.DataFrame([{
        "Model Type": "DefakeHop++ Approx",
        "mins": total_mins,
        "hrs": total_mins / 60.0
    }])


In [ ]:
def plot_resolution_results(res_df):
    tmp = res_df.copy()
    tmp["res"] = tmp["Resolution"].str.split("x").str[0].astype(int)
    tmp = tmp.sort_values("res")

    plt.figure(figsize=(8, 4))
    plt.plot(tmp["res"], tmp["Accuracy"], marker="o", label="Accuracy")
    plt.plot(tmp["res"], tmp["AUC"], marker="s", label="AUC")
    plt.plot(tmp["res"], tmp["Avg Uncertainty"], marker="^", label="Avg Uncertainty")
    plt.xlabel("Resolution")
    plt.ylabel("Score")
    plt.title("DefakeHop++ Approx vs Resolution")
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
model, timing = fit_defakehoppp_pipeline(
    train_df=train_df,
    val_df=val_df,
    image_size=224
)

timing

In [ ]:
test_results_df = evaluate_model(
    model,
    test_df,
    image_size=224,
    model_name="DefakeHop++ Approx"
)

test_results_df

In [ ]:
res_df = evaluate_resolutions(
    model,
    test_df,
    resolutions=(128, 224, 256, 384)
)

res_df

In [ ]:
inf_df = measure_inference_time_by_resolution(
    model,
    test_df,
    resolutions=(128, 224, 256, 384),
    max_samples=500
)

inf_df

In [ ]:
train_time_df = make_training_time_table(timing)
train_time_df

In [ ]:
plot_resolution_results(res_df)

In [ ]:
os.makedirs(cfg.out_dir, exist_ok=True)

test_results_df.to_csv(os.path.join(cfg.out_dir, "test_results.csv"), index=False)
res_df.to_csv(os.path.join(cfg.out_dir, "accuracy_based_on_resolution.csv"), index=False)
inf_df.to_csv(os.path.join(cfg.out_dir, "inference_time.csv"), index=False)
train_time_df.to_csv(os.path.join(cfg.out_dir, "training_time.csv"), index=False)
pd.DataFrame([timing]).to_csv(os.path.join(cfg.out_dir, "training_stage_timing_breakdown.csv"), index=False)

print("Saved to:", cfg.out_dir)